# Case Study §1 — Build the CPT Corpus from Wikipedia (computational / quantum chemistry)

This notebook is the **runnable, verifiable twin** of [`01_build_corpus.py`](01_build_corpus.py). It imports the
script's functions directly (single source of truth — notebook and script can't drift), demonstrates the
tricky bits live, runs the full build, and then **asserts** the output is clean.

### Why a raw-text corpus (and what it implies for loss)
Continued pretraining (CPT) is plain next-token prediction over raw text, so the training signal is the
**full causal LM loss on *every* token**. That is *why* this corpus is a stream of raw text (`{"text": ...}`)
and not prompt/completion pairs — there is no prompt to mask. Contrast with SFT (§5/§6), where data is
prompt→completion and loss is computed on the **completion only** (`completion_only_loss=True`). The *shape*
of the data here is dictated by the loss we intend to compute. (See `RESEARCH_NOTES.md` §7.)

### The recipe (every setting is deliberate — reproducibility is a hard requirement)
| Step | Choice | Why (research-backed) |
|---|---|---|
| Source | MediaWiki API `prop=extracts&explaintext` | clean prose; CC BY-SA (attributed in `manifest.json`) |
| User-Agent | descriptive UA string | Wikipedia returns **HTTP 403** to absent/bare UAs |
| Math | `{\displaystyle …}` → inline `$ … $`, glyph-soup removed | `explaintext` renders each equation twice (glyph dump + LaTeX); we keep the LaTeX |
| Tokens | **add no new vocab tokens** | LaTeX is ASCII; existing BPE handles it. Adding needless special tokens hurts performance (HF). CPT adapts existing embeddings (→ `embed_tokens`+`lm_head` in LoRA targets) |
| Normalize | Unicode **NFKC** | canonicalize odd code points |
| Dedup | resolved-title + content-hash | redirects silently duplicate pages; duplication over-weights topics & wastes compute (D4, arXiv:2308.12284) |
| Min length | 200 chars | drop disambiguation / wrong-title stubs |


In [ ]:
# Load the script's functions (it starts with a digit, so import via importlib) + seed + env.
import importlib.util, pathlib, sys, json

HERE = pathlib.Path.cwd()
script = HERE / "01_build_corpus.py"
if not script.exists():
    script = HERE / "case_study" / "01_build_corpus.py"   # if launched from repo root
sys.path.insert(0, str(script.parent))
spec = importlib.util.spec_from_file_location("corpus_builder", script)
cb = importlib.util.module_from_spec(spec); spec.loader.exec_module(cb)
import config

config.set_all_seeds()
print(json.dumps(config.record_env(), indent=2))

## The math problem, demonstrated
Wikipedia's `explaintext` renders an equation **twice**: a broken glyph-by-glyph Unicode dump (one symbol
per line) followed by the clean LaTeX source inside `{\displaystyle …}`. Feeding the glyph soup into CPT would
train the model on noise. Let's run `clean()` on a tiny synthetic sample shaped exactly like the real thing
and **verify** the soup is removed and the LaTeX is kept.

In [ ]:
sample = (
    "The time-dependent equation is:\n"
    " i\n ℏ\n ∂\n Ψ\n =\n H\n Ψ\n"                       # <- glyph-soup duplicate
    " {\\displaystyle i\\hbar \\partial_t \\Psi = H \\Psi }\n"  # <- clean LaTeX source
    "where H is the Hamiltonian.\n"
)
cleaned, n_eq = cb.clean(sample)
print("--- BEFORE ---\n", repr(sample))
print("\n--- AFTER ---\n", cleaned)

assert "displaystyle" not in cleaned, "LaTeX wrapper should be gone"
assert "$ i\\hbar \\partial_t \\Psi = H \\Psi $" in cleaned, "equation should survive as inline $...$"
assert "ℏ\n" not in cleaned, "glyph-soup duplicate should be removed"
assert n_eq == 1
print("\n✓ math handling verified: 1 equation kept as LaTeX, glyph soup removed")

## Build the full corpus
`build_corpus()` fetches (or loads from cache) every page in `config.WIKI_PAGES`, applies the recipe above,
dedups, and writes `data/corpus/*.txt` + `manifest.json`. Re-running is cheap (cached); pass `force=True` to refetch.

In [ ]:
manifest = cb.build_corpus(force=False)

## Inspect & verify the corpus
Show the per-page stats, the totals, and **assert** the quality guarantees the chapter will rely on.

In [ ]:
pages = manifest["pages"]
tot_words = sum(p["words"] for p in pages)
tot_eq = sum(p["equations"] for p in pages)
print(f"{len(pages)} unique pages | ~{tot_words:,} words | ~{tot_words*4//3:,} tokens est. | {tot_eq:,} equations\n")
top = sorted(pages, key=lambda p: p['equations'], reverse=True)[:8]
print('Most equation-dense pages:')
for p in top:
    print(f"  {p['resolved']:42} {p['words']:>6} words  {p['equations']:>4} eqs")

# Reproducibility / quality assertions
assert len(pages) >= 35, 'expected a few dozen unique pages'
assert tot_eq > 500, 'equation handling should preserve hundreds of equations'
files = list(config.CORPUS_DIR.glob('*.txt'))
import collections
hashes = collections.Counter(p['sha256'] for p in pages)
assert all(c == 1 for c in hashes.values()), 'no duplicate content hashes should remain'
# spot-check a known math page: clean LaTeX, no leftover wrappers
schro = (config.CORPUS_DIR / 'schr_dinger_equation.txt').read_text()
assert 'displaystyle' not in schro and schro.count(' $ ') > 100
print('\n✓ verified: unique pages, equations preserved, no duplicates, clean LaTeX')

## What this teaches — and what's next
- **Data availability is asymmetric.** We just produced ~100k+ tokens of clean domain text in seconds, with
  zero labelling. That is the *abundant* side. `02_data_availability.py` quantifies the *scarce* side — how
  little instruction (SFT) data you can realistically hand-build for the same domain.
- **Data shape ↔ loss.** This raw-text corpus is for CPT → **full causal loss on all tokens**. SFT data will be
  prompt→completion → **loss on the completion only**.
- **Equations are content, not noise.** We kept 1k+ equations as LaTeX and added no special tokens; CPT will adapt
  the existing embeddings to this math-heavy distribution.

**Next:** `02_data_availability.py` (SFT-data scarcity), then `03_cpt.py` (continued pretraining).